# 08 — PCA + Random Forest baseline

Can one pixel's spectrum identify mock stress as well as the HybridSN patch classifier? Run notebooks 03, 04 and 06 first. Reuse the train-fitted transform and checkpoint from notebook 06. Compare both models on every valid centre from row 38 onward. The split cuts through the field's only simulated stress spot.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

root = Path(".") if Path("notebooks").is_dir() else Path("..")
cube_file = root / "data/mock/mock_cube.h5"
pca_file = root / "data/processed/mock_pca.npz"
split_row = 38  # First validation row in notebook 06

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_fscore_support

ckpt = torch.load(root / "models/hybridsn_mock.pt", weights_only=True, map_location="cpu")
with np.load(pca_file) as archive:
    valid = archive["valid"]
    expected_labels = archive["labels"]
with h5py.File(cube_file) as source:
    wavelengths = source["wavelengths_nm"][...]
    bands = np.isin(wavelengths, ckpt["wavelengths_nm"].numpy())
    assert np.array_equal(wavelengths[bands], ckpt["wavelengths_nm"].numpy())
    labels = source["labels"][...]
    assert np.array_equal(labels, expected_labels)
    spectra = source["cube"][...][..., bands]

# Notebook 06 saves the band scaling, PCA basis and whitening denominators.
X = spectra[valid].astype(np.float64)
X = (X - ckpt["band_mean"].numpy()) / ckpt["band_scale"].numpy()
X = (X - ckpt["pca_mean"].numpy()) @ ckpt["pca_components"].numpy().T
X = (X / ckpt["pca_scale"].numpy()).astype(np.float32)
rows, cols = np.nonzero(valid)
train = rows < split_row
y = labels[rows, cols]
for name, mask in (("Training", train), ("Validation", ~train)):
    healthy, stressed = np.bincount(y[mask], minlength=2)
    assert healthy > 0 and stressed > 0
    print(f"{name}: {healthy} healthy, {stressed} stressed")
assert np.isfinite(X).all()

Training: 2538 healthy, 8 stressed
Validation: 1930 healthy, 13 stressed


### Train the pixel baseline

Use all valid training pixels and their ten whitened PCA features. Fix the seed and use 200 bootstrap trees with balanced class weights computed on the training set. Use the forest's stressed-class probability above 0.5 to flag a centre. These settings are fixed before evaluating the validation rows.

In [2]:
forest = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0, n_jobs=1)
forest.fit(X[train], y[train])
y_val = y[~train]
rf_score = forest.predict_proba(X[~train])[:, forest.classes_ == 1].ravel()

### Score the saved HybridSN on the same centres

Reconstruct its validation patches from the same transformed features. Use notebook 06's dataset, model architecture, reflection padding and saved logit threshold. Check the row and column order before comparing predictions.

In [3]:
class PatchDataset(Dataset):
    def __init__(self, windows, centres, labels):
        self.windows = windows            # (H, W, K, P, P) view, no copy
        self.centres = centres            # (n, 2) array of (row, col)
        self.labels = labels              # (n,) 0 = healthy, 1 = stressed

    def __len__(self):
        return len(self.centres)

    def __getitem__(self, i):
        r, c = self.centres[i]
        patch = torch.from_numpy(np.ascontiguousarray(self.windows[r, c]))     # (K, P, P)
        return patch.unsqueeze(0), torch.tensor(float(self.labels[i]))


class HybridSNMock(nn.Module):
    def __init__(self, n_bands=10, patch=9, dropout=0.4):
        super().__init__()
        self.spectral_spatial = nn.Sequential(                 # 3D part
            nn.Conv3d(1, 8, kernel_size=3), nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=3), nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=3), nn.ReLU(),
        )
        with torch.no_grad():                                  # measure the shape instead of computing it by hand
            d, h, w = self.spectral_spatial(torch.zeros(1, 1, n_bands, patch, patch)).shape[2:]

        self.spatial = nn.Sequential(                          # 2D part
            nn.Conv2d(32 * d, 64, kernel_size=3), nn.ReLU(),
        )
        with torch.no_grad():
            flat = self.spatial(torch.zeros(1, 32 * d, h, w)).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),                                  # ONE logit: "stressed"
        )

    def forward(self, x, verbose=False):
        show = (lambda name, t: print(f"{name:<28}{tuple(t.shape)}")) if verbose else (lambda name, t: None)
        show("input", x)
        x = self.spectral_spatial(x)                           # (N, 32, D', H', W')
        show("after 3 x Conv3d", x)
        n, c, d, h, w = x.shape
        x = x.reshape(n, c * d, h, w)                          # merge channels and bands
        show("reshaped for 2D", x)
        x = self.spatial(x)
        show("after Conv2d", x)
        x = self.classifier(x).squeeze(1)
        show("output (one logit each)", x)
        return x

In [4]:
val_valid = valid[split_row:]
val_features = np.zeros((*val_valid.shape, ckpt["n_bands"]), dtype=np.float32)
val_features[val_valid] = X[~train]
patch = ckpt["patch"]
r = patch // 2
padded = np.pad(val_features, ((r, r), (r, r), (0, 0)), mode="reflect")
windows = np.lib.stride_tricks.sliding_window_view(padded, (patch, patch), axis=(0, 1))
centres = np.argwhere(val_valid)
assert np.array_equal(centres + [split_row, 0], np.column_stack((rows[~train], cols[~train])))
val_set = PatchDataset(windows, centres, y_val)
cnn = HybridSNMock(n_bands=ckpt["n_bands"], patch=patch)
cnn.load_state_dict(ckpt["state_dict"])
cnn.eval()
with torch.inference_mode():
    cnn_score = torch.cat([cnn(xb) for xb, _ in DataLoader(val_set, batch_size=64)]).numpy()

### Compare decisions and ranking

Report confusion counts and metrics for the stressed class. Average precision (AP) uses continuous scores across thresholds. Prevalence gives a reference for an uninformative ranking.

In [5]:
print(f"{'Model':16} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>5} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AP':>8}")
for name, score, threshold in [
    ("PCA + RF", rf_score, 0.5),
    ("HybridSN", cnn_score, ckpt["threshold_logit"]),
]:
    assert score.shape == y_val.shape and np.isfinite(score).all()
    pred = score > threshold
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(y_val, pred, average="binary", zero_division=0)
    ap = average_precision_score(y_val, score)
    print(f"{name:16} {tp:4} {fp:4} {fn:4} {tn:5} {precision:10.3f} {recall:8.3f} {f1:8.3f} {ap:8.3f}")
print(f"Validation prevalence: {y_val.mean():.3%} ({int(y_val.sum())}/{len(y_val)})")

Model              TP   FP   FN    TN  Precision   Recall       F1       AP
PCA + RF           13    0    0  1930      1.000    1.000    1.000    1.000
HybridSN           13   59    0  1871      0.181    1.000    0.306    1.000
Validation prevalence: 0.669% (13/1943)


### Reading the result

On these 1,943 validation centres the forest detected all 13 stressed pixels with zero false alarms. HybridSN found the same 13 and also flagged 59 healthy pixels. Both models achieved average precision of 1.000, so their rankings separated the classes even though their fixed thresholds produced different alerts. The centre-pixel spectrum was sufficient on this split. With eight stressed training centres and thirteen validation positives from the same synthetic spot, field-to-field performance remains unmeasured.